# Experiment Runner Workshop

```
make_arithmetic.py
    generates data/arithmetic.jsonl, our dataset

build_experiment.py
    generates experiment.csv, our dataframe (saved as CSV)
    each prompting strategy (condition) is a row

runner.py
    reads experiment.csv, and expands this dataframe
    into a certain number of individual tasks

    produces a runs.csv (record of runs)
             and results/{run_id}.jsonl for each run
             (contains each run's output)

collate.py
    reads runs.csv to understand what has been done,
    reads each run's corresponding jsonl output, and
    builds a dataframe of this experiment's outputs
```

## 1: Data generation

The dataset is a list of grade-school word problems. Each item is a dict with a `question`, the integer `answer`, how many arithmetic `steps` it took, and the `n_steps` count. Let's load it and look at one.

In [1]:
import json
from make_arithmetic import generate_arithmetic

dataset = [
    generate_arithmetic(n_steps=n, seed=i * 7 + n)
    for n in range(2, 5)  # 2, 3, 4 steps
    for i in range(34)
]  # 34 samples per step count
# 102 total samples

with open("data/arithmetic.jsonl", "w") as f:
    for item in dataset:
        f.write(json.dumps(item) + "\n")

print(f"Wrote {len(dataset)} samples to data/arithmetic.jsonl")

Wrote 102 samples to data/arithmetic.jsonl


In [2]:
with open("data/arithmetic.jsonl") as f:
    dataset = [json.loads(line) for line in f]

print(f"{len(dataset)} items")
dataset[0]

102 items


{'question': 'Ursula has 7 apples. Ursula spends 2 apples. Ursula donates 2 apples. How many apples does Ursula have now?',
 'answer': 3,
 'n_steps': 2,
 'steps': [-2, -2]}

**Predict the output.** The generator seeds Python's RNG from its `seed` argument. We call it twice with the *same* seed below. What will `a == b` be?

In [3]:
from make_arithmetic import generate_arithmetic

a = generate_arithmetic(n_steps=3, seed=42)
b = generate_arithmetic(n_steps=3, seed=42)

print("identical:", a == b)
a["question"]

identical: True


'Lidia has 5 oranges. Lidia buys 4 more oranges. Lidia donates 2 oranges. Lidia gives away 4 oranges. How many oranges does Lidia have now?'

## 2: The PLAN stage — registries + sweeps

`TASK_SWEEPS`: a dictionary, each key represents a task (`arithmetic`, `coin_flip`) and the value contains relevant parameters (`condition`, `n_samples`).

`LLM_SWEEP`: a dictionary, containing the LLMs and any relevant LLM parameters.

Want to compute the cartesian product of these: all tasks cross all LLMs.

In [4]:
from build_experiment import TASK_SWEEPS, LLM_SWEEP

print("TASK_SWEEPS:", TASK_SWEEPS)
print("LLM_SWEEP:  ", LLM_SWEEP)

TASK_SWEEPS: {'arithmetic': {'condition': ['standard', 'zs_cot'], 'n_samples': [10]}}
LLM_SWEEP:   {'llm': ['qwen2.5-7b'], 'temperature': [0.0], 'top_k': [3]}


In [31]:
import itertools


def plan_conditions(task_sweeps, llm_sweep):
    """
    Cross every task's arg sweep with the LLM sweep.

    Pattern:
        Create a list, containing all combinations (using itertools.product) of the LLM sweep (there is only 1) and the task sweep (there are multiple).
    """
    llm_keys = list(llm_sweep)
    llm_combos = list(itertools.product(*(llm_sweep[k] for k in llm_keys)))

    rows = []
    for task, sweep in task_sweeps.items():
        keys = list(sweep)
        for combo in itertools.product(*(sweep[k] for k in keys)):
            task_cfg = dict(zip(keys, combo))
            for llm_combo in llm_combos:
                llm_cfg = dict(zip(llm_keys, llm_combo))
                rows.append({"task": task, "task_cfg": task_cfg, "llm_cfg": llm_cfg})
    return rows


plan_conditions(TASK_SWEEPS, LLM_SWEEP)

[{'task': 'arithmetic',
  'task_cfg': {'condition': 'standard', 'n_samples': 10},
  'llm_cfg': {'llm': 'qwen2.5-7b', 'temperature': 0.0, 'top_k': 3}},
 {'task': 'arithmetic',
  'task_cfg': {'condition': 'zs_cot', 'n_samples': 10},
  'llm_cfg': {'llm': 'qwen2.5-7b', 'temperature': 0.0, 'top_k': 3}}]

**Predict the output.** With the sweeps above (one task, two conditions,
one LLM config), how many condition rows should `build()` produce? Now import
the real thing and check — it adds seeds, repeats, and git provenance on top of
the product you just wrote.

In [ ]:
import build_experiment

build_experiment.main()  # writes experiment.csv
plan = build_experiment.build()
plan[["idx", "task", "task_args", "llm_args", "repeats", "base_seed"]]

## 3: Why JSON-in-CSV?

A CSV cell can't hold a Python dict, so `task_args` and `llm_args` are stored as **JSON strings**.
They're split into two columns on purpose: `task_args` fork to the task's `run()` and `llm_args` fork to the LLM constructor.

**Predict the output.** What is the *type* of the cell below before and after `json.loads`?

In [ ]:
cell = plan["task_args"].iloc[0]
print("raw:   ", type(cell).__name__, "->", repr(cell))

import json

parsed = json.loads(cell)
print("parsed:", type(parsed).__name__, "->", parsed)

## 4: The RUN stage

The plan has one row per *condition*.
Running it means expanding each condition into `repeats` separate executions, each with its own `run_id` and `seed`.
That seed is what makes a single run reproducible.

In [ ]:
def expand_runs(plan):
    """Expand each condition into `repeats` executions."""
    rows = []
    for _, cond in plan.iterrows():
        for k in range(int(cond["repeats"])):
            rows.append(
                {
                    "run_id": f"run{int(cond['idx']):02d}_rep{k}",
                    "condition_idx": int(cond["idx"]),
                    "repeat_idx": k,
                    "seed": int(cond["base_seed"]) + k,
                }
            )
    return rows


expand_runs(plan)

Now run it for real against the live Qwen endpoint. This calls the model, so it
takes a few seconds per run.

In [ ]:
from runner import run_all

runs = run_all("experiment.csv")  # live LLM calls happen here
runs[["run_id", "task", "accuracy", "runtime_s", "results_path"]]

## 5: The Gap

Look at `runs.csv`: the `results_path` column only **points** at the per-sample files. The actual question-by-question answers live in `results/*.jsonl`.
Nothing has loaded them back into a dataframe yet — so "everything is a dataframe" breaks right here, at the analysis step.

In [ ]:
import pandas as pd

runs = pd.read_csv("runs.csv")
print(runs[["run_id", "results_path", "accuracy"]].to_string(index=False))

print("\nOne line from the first run's results file:")
with open(runs["results_path"].iloc[0]) as f:
    print(f.readline().strip())

## 6: Build `collate`

The missing stage.
Goal: **one row per question**, with the run's metadata joined onto every row, so a single tidy table feeds all our plots.

In [ ]:
import json
import pandas as pd

RUN_META = ["run_id", "condition_idx", "repeat_idx", "seed", "task", "llm", "runtime_s"]


def collate(runs_path="runs.csv"):
    """Load every run's per-sample jsonl into one tidy dataframe."""
    runs = pd.read_csv(runs_path)

    frames = []
    for _, run in runs.iterrows():
        with open(run["results_path"]) as f:
            samples = pd.DataFrame(json.loads(line) for line in f)
        for col in RUN_META:
            samples[col] = run[col]
        frames.append(samples)

    df = pd.concat(frames, ignore_index=True)
    df["correct"] = df["prediction"] == df["answer"]
    return df


df = collate()
df.head()

The true version lives in **`collate.py`**.
It does the same join, plus one extra move: it explodes the `task_args`/`llm_args` JSON back into real
`arg_*` / `llm_*` columns (the same trick `build_experiment.explode_task_args`
uses).

We'll use those columns to group by condition next.

In [ ]:
import importlib, collate as collate_mod

importlib.reload(collate_mod)

df = collate_mod.collate()
print("columns:", list(df.columns))
df.head()

## 7: Analysis

One tidy dataframe, three `groupby`s. **Predict each before you run it.**

### Accuracy by condition
Does "let's think step by step" (`zs_cot`) actually beat the bare `standard`
prompt?

In [ ]:
import matplotlib.pyplot as plt

by_cond = df.groupby("arg_condition")["correct"].mean()
print(by_cond)
by_cond.plot.bar(title="Accuracy by condition", ylabel="accuracy", rot=0)
plt.tight_layout()
plt.show()

### Accuracy by difficulty
`n_steps` is how many arithmetic operations the problem chains. Do you expect
accuracy to fall as the problems get longer?

In [ ]:
by_steps = df.groupby("n_steps")["correct"].mean()
print(by_steps)
by_steps.plot.bar(title="Accuracy by difficulty (n_steps)", ylabel="accuracy", rot=0)
plt.tight_layout()
plt.show()

### Runtime by condition
`zs_cot` makes the model generate a whole chain of reasoning. What does that do
to wall-clock time per run?

In [ ]:
by_runtime = (
    df.groupby(["arg_condition", "run_id"])["runtime_s"]
    .first()
    .groupby("arg_condition")
    .mean()
)
print(by_runtime)
by_runtime.plot.bar(title="Mean runtime by condition", ylabel="seconds", rot=0)
plt.tight_layout()
plt.show()

## 8: Extra Time, Add the "Coin Flip" Task

`tasks/coin_flips.py` is already fully implemented, and `runner.py` already registers it.
The *only* thing stopping it is two commented-out blocks in **`build_experiment.py`**:

1. the `"coin_flips"` entry in `TASK_REGISTRY`, and
2. the `"coin_flips"` block in `TASK_SWEEPS`.

**Your task:**

1. Open `build_experiment.py` and uncomment both blocks.
2. Re-run the whole pipeline: `build` → `run_all` → `collate`.
3. Re-plot accuracy **grouped by task as well as condition**.

Notice what you *don't* have to touch: `collate` never mentioned `n_steps` or any task-specific column, so coin_flips (which has no `n_steps`) slots straight into the tidy table — NaN where it doesn't apply.

In [ ]:
import importlib
import build_experiment, runner, collate as collate_mod

importlib.reload(build_experiment)
importlib.reload(collate_mod)

build_experiment.main()
runner.run_all("experiment.csv")
df = collate_mod.collate()

import matplotlib.pyplot as plt

acc = df.groupby(["task", "arg_condition"])["correct"].mean().unstack("arg_condition")
print(acc)
acc.plot.bar(title="Accuracy by task and condition", ylabel="accuracy", rot=0)
plt.tight_layout()
plt.show()